# Meta Quest VR Collection Dashboard

이 노트북은 `/home/hyunjin/rby1_ws/rby1-data-collection/main.py` 에서 실행 중인 Meta Quest VR data collection backend에 붙는 얇은 대시보드입니다. 로봇 제어 로직을 여기서 직접 실행하는 것이 아니라, backend의 상태를 조회하고 녹화 명령을 보내는 용도입니다.

## 이 노트북에서 할 수 있는 것
- 현재 recording status 확인
- preflight readiness 확인
- Torso Mode / Bimanual Mode ready pose 요청
- 카메라 진단 상태와 preview availability 확인
- 현재 저장 경로, 마지막 저장 경로, 마지막 discard 경로 확인
- `Start`, `Stop & Save`, `Discard Last`, `Refresh` 명령 전송

## 권장 실행 순서
1. 별도 터미널에서 backend를 먼저 실행합니다.
   `python /home/hyunjin/rby1_ws/rby1-data-collection/main.py --data_collect --dashboard_cmd_server tcp://127.0.0.1:5556`
2. 이 노트북에서 아래 셀들을 위에서 아래로 순서대로 실행합니다.
3. 중간에 endpoint를 바꿨다면 설정 셀만 다시 실행합니다.
4. UI 셀이 뜨면 `Init Torso` 또는 `Init Bimanual`로 로봇 ready pose를 먼저 요청합니다.
5. `Check Cameras` 또는 `Refresh`로 camera panel과 preview를 확인합니다.
6. preflight가 준비되면 `Start`로 녹화를 시작하고, 종료 시 `Stop & Save`를 누릅니다.
7. 직전 데모를 버리려면 `Discard Last`를 사용합니다.

## 상태 해석
- `IDLE`은 backend가 살아 있지만 녹화를 아직 시작하지 않았다는 뜻입니다.
- `Ready pose`가 `WAIT`이면 아직 초기 자세 요청이 안 갔거나 로봇 이동이 끝나지 않은 상태입니다.
- preview가 비어 있으면 camera diagnostics 패널의 `Live frames` 와 `Missing live`를 먼저 확인하면 됩니다.

## 셀 구성
아래에 각 코드 셀의 역할을 설명하는 markdown을 붙여 두었습니다. 처음 보는 사람도 어떤 셀을 왜 실행하는지 바로 알 수 있게 정리한 것입니다.

## 셀 1. 기본 import와 backend client 로드

이 셀은 노트북이 동작하는 데 필요한 Python 모듈을 import하고, `rby1-data-collection` 경로를 `sys.path`에 추가한 뒤 dashboard client helper를 불러옵니다.

이 셀을 먼저 실행해야 이후 셀에서 `VRDashboardClient`, 상태 렌더링 함수, 위젯 구성이 정상적으로 동작합니다.

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any
import sys
import threading
import time

import ipywidgets as widgets
from IPython.display import display

REPO_ROOT = Path("/home/hyunjin/rby1_ws/rby1-data-collection")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vr_dashboard_client import (
    VRDashboardClient,
    render_camera_html,
    render_paths_html,
    render_preflight_html,
    render_status_badge,
)

print(f"Repo root: {REPO_ROOT}")

Repo root: /home/hyunjin/rby1_ws/rby1-data-collection


## 셀 2. 연결 endpoint와 refresh 동작 설정

이 셀은 backend REP 서버 주소, 요청 타임아웃, 자동 새로고침 주기, preview 포함 여부 같은 런타임 설정을 정의합니다.

backend를 다른 포트나 다른 호스트에서 띄웠다면 여기의 `COMMAND_ENDPOINT`를 먼저 바꾸고 실행하면 됩니다.

In [2]:
COMMAND_ENDPOINT = "tcp://127.0.0.1:5556"
REQUEST_TIMEOUT_MS = 4000
AUTO_REFRESH_SECS = 1.0
INCLUDE_PREVIEW = True
PREVIEW_MAX_WIDTH = "480px"

client = VRDashboardClient(COMMAND_ENDPOINT, timeout_ms=REQUEST_TIMEOUT_MS)
print(f"Dashboard endpoint: {COMMAND_ENDPOINT}")

Dashboard endpoint: tcp://127.0.0.1:5556


## 셀 3. 대시보드 세션 상태 보관

이 셀은 notebook 쪽에서만 쓰는 `DashboardState`를 정의합니다. 마지막으로 받아온 snapshot, auto refresh 스레드, 중지 이벤트를 한 곳에 묶어 둡니다.

UI를 그리기 전에 이 상태 객체가 있어야 이후 helper 함수들이 공통 상태를 안전하게 공유할 수 있습니다.

In [3]:
@dataclass(slots=True)
class DashboardState:
    last_snapshot: dict[str, Any] = field(default_factory=dict)
    auto_refresh_enabled: bool = False
    auto_refresh_stop: threading.Event = field(default_factory=threading.Event)
    auto_refresh_thread: threading.Thread | None = None


dashboard_state = DashboardState()
dashboard_state

DashboardState(last_snapshot={}, auto_refresh_enabled=False, auto_refresh_stop=<threading.Event at 0x7b13705134d0: unset>, auto_refresh_thread=None)

## 셀 4. 렌더링과 command helper 정의

이 셀은 preview HTML 생성, backend snapshot 조회, 상태 패널 업데이트, `start`/`stop`/`discard` 명령 전송, auto refresh 루프를 담당하는 핵심 helper 함수들을 정의합니다.

실제 버튼 동작은 다음 UI 셀에서 연결되지만, 그 기반 로직은 모두 여기에서 준비됩니다.

In [4]:
def render_preview_html(snapshot: dict[str, Any]) -> str:
    encoded = snapshot.get("preview_jpeg_base64")
    if not encoded:
        camera_diagnostics = snapshot.get("camera_diagnostics", {})
        detail = camera_diagnostics.get("message") or snapshot.get("preflight", {}).get("camera_message") or "No live camera frame is available yet."
        return (
            "<div style='padding:14px;border:1px dashed #9fb3c8;border-radius:12px;color:#486581;background:#f8fbfc;'>"
            "<div style='font-weight:700;margin-bottom:4px;'>Preview unavailable</div>"
            f"<div style='font-size:13px;'>{detail}</div>"
            "</div>"
        )
    return (
        "<div style='padding:6px;border:1px solid #d9e2ec;border-radius:12px;background:#ffffff;'>"
        f"<img src='data:image/jpeg;base64,{encoded}' style='width:100%;max-width:{PREVIEW_MAX_WIDTH};border-radius:10px;'/>"
        "</div>"
    )


def set_log(message: str):
    log_html.value = (
        "<div style='padding:8px 12px;border-radius:10px;background:#eef6ff;color:#102a43;'>"
        f"{message}"
        "</div>"
    )


def fetch_snapshot(include_preview: bool = INCLUDE_PREVIEW) -> dict[str, Any]:
    snapshot = client.get_status(include_preview=include_preview)
    dashboard_state.last_snapshot = snapshot
    return snapshot


def update_dashboard(snapshot: dict[str, Any]):
    has_state = isinstance(snapshot.get("preflight"), dict) and "status" in snapshot

    if snapshot.get("ok", False) or has_state:
        status_message = snapshot.get("message", "")
        if not snapshot.get("ok", False) and snapshot.get("error"):
            status_message = f"{status_message} | {snapshot['error']}" if status_message else snapshot["error"]

        status_html.value = render_status_badge(snapshot.get("status", "unknown"), status_message)
        preflight_html.value = render_preflight_html(snapshot.get("preflight", {}))
        camera_html.value = render_camera_html(snapshot.get("camera_diagnostics", {}))
        paths_html.value = render_paths_html(snapshot)
        preview_html.value = render_preview_html(snapshot)
        raw_state.value = repr({k: v for k, v in snapshot.items() if k != "preview_jpeg_base64"})
    else:
        error_text = snapshot.get("error", "Unknown backend error")
        status_html.value = render_status_badge("error", error_text)
        preflight_html.value = render_preflight_html({"message": error_text})
        camera_html.value = render_camera_html({"message": error_text})
        paths_html.value = "<div style='padding:10px 14px;color:#9b2226;'>No session data available.</div>"
        preview_html.value = render_preview_html({})
        raw_state.value = repr(snapshot)


def refresh_dashboard(_=None):
    try:
        snapshot = fetch_snapshot(include_preview=INCLUDE_PREVIEW)
        update_dashboard(snapshot)
        set_log(f"Refreshed at {time.strftime('%H:%M:%S')}")
    except Exception as exc:
        update_dashboard({"ok": False, "error": str(exc)})
        set_log(f"Refresh failed: {exc}")


def run_command(command_name: str):
    try:
        if command_name == "start":
            snapshot = client.start_recording()
        elif command_name == "stop":
            snapshot = client.stop_recording()
        elif command_name == "discard":
            snapshot = client.discard_last_demo()
        elif command_name == "init_torso":
            snapshot = client.init_torso_mode()
        elif command_name == "init_bimanual":
            snapshot = client.init_bimanual_mode()
        elif command_name == "check_cameras":
            snapshot = client.check_cameras()
        else:
            raise ValueError(f"Unsupported command: {command_name}")

        dashboard_state.last_snapshot = snapshot
        update_dashboard(snapshot)

        command_message = snapshot.get("command_message")
        if not command_message:
            command_message = f"Command '{command_name}' sent at {time.strftime('%H:%M:%S')}"
        elif command_name in {"init_torso", "init_bimanual"}:
            command_message = f"{command_message} Refresh or Auto Refresh while the robot moves."

        set_log(command_message)
    except Exception as exc:
        update_dashboard({"ok": False, "error": str(exc)})
        set_log(f"Command '{command_name}' failed: {exc}")


def stop_auto_refresh():
    dashboard_state.auto_refresh_enabled = False
    dashboard_state.auto_refresh_stop.set()


def start_auto_refresh():
    if dashboard_state.auto_refresh_thread is not None and dashboard_state.auto_refresh_thread.is_alive():
        return

    dashboard_state.auto_refresh_enabled = True
    dashboard_state.auto_refresh_stop.clear()

    def _loop():
        while not dashboard_state.auto_refresh_stop.is_set():
            refresh_dashboard()
            dashboard_state.auto_refresh_stop.wait(AUTO_REFRESH_SECS)

    dashboard_state.auto_refresh_thread = threading.Thread(target=_loop, daemon=True)
    dashboard_state.auto_refresh_thread.start()

## 셀 5. 위젯 UI 생성과 첫 상태 조회

이 셀은 ready pose 버튼, camera check 버튼, recording 버튼, 상태 패널, camera diagnostics, preview, raw state 영역을 실제 `ipywidgets` 레이아웃으로 만들고 화면에 표시합니다. 셀 마지막의 `refresh_dashboard()`가 첫 backend 상태 조회를 즉시 수행합니다.

추가로 UI 상단에 각 dashboard 버튼이 Meta Quest VR 컨트롤러의 어떤 버튼과 대응되는지 한눈에 볼 수 있는 매핑 박스도 함께 표시합니다.

보통 사용자는 여기까지 실행한 뒤 `Init Torso` 또는 `Init Bimanual`로 초기 자세를 맞추고, camera panel을 확인한 뒤 녹화를 시작하게 됩니다.

In [5]:
title_html = widgets.HTML("<h2 style='margin:4px 0'>Meta Quest VR Data Collection</h2>")
subtitle_html = widgets.HTML(
    "<div style='color:#486581;font-size:13px;'>Ready pose, camera diagnostics, and recording controls for the backend command API in main.py</div>"
)
vr_mapping_html = widgets.HTML(
    """
    <div style='padding:10px 14px;border:1px solid #d9e2ec;border-radius:12px;background:#fffaf0;'>
      <div style='font-size:12px;color:#7c5e10;margin-bottom:6px;'>VR CONTROLLER MAP</div>
      <table style='border-collapse:collapse;font-size:13px;width:100%;'>
        <tr><td style='padding:4px 8px;'><b>Init Torso</b></td><td style='padding:4px 8px;'>Right controller A</td></tr>
        <tr><td style='padding:4px 8px;'><b>Init Bimanual</b></td><td style='padding:4px 8px;'>Left controller Y</td></tr>
        <tr><td style='padding:4px 8px;'><b>Start</b></td><td style='padding:4px 8px;'>Left controller X toggle from IDLE</td></tr>
        <tr><td style='padding:4px 8px;'><b>Stop &amp; Save</b></td><td style='padding:4px 8px;'>Right controller B or Left controller X toggle while recording</td></tr>
        <tr><td style='padding:4px 8px;'><b>Check Cameras</b></td><td style='padding:4px 8px;'>Dashboard only</td></tr>
        <tr><td style='padding:4px 8px;'><b>Refresh / Auto Refresh</b></td><td style='padding:4px 8px;'>Dashboard only</td></tr>
        <tr><td style='padding:4px 8px;'><b>Discard Last</b></td><td style='padding:4px 8px;'>Dashboard only</td></tr>
      </table>
      <div style='font-size:12px;color:#486581;margin-top:8px;'>
        Hold the matching controller grip button to follow that arm. Use the matching trigger to open or close each gripper.
      </div>
    </div>
    """
)
status_html = widgets.HTML()
preflight_html = widgets.HTML()
camera_html = widgets.HTML()
paths_html = widgets.HTML()
preview_html = widgets.HTML()
log_html = widgets.HTML()
raw_state = widgets.Textarea(
    value="",
    description="Raw",
    layout=widgets.Layout(width="100%", height="180px"),
)

btn_init_torso = widgets.Button(description="Init Torso", button_style="primary", icon="send")
btn_init_bimanual = widgets.Button(description="Init Bimanual", button_style="primary", icon="send")
btn_check_cameras = widgets.Button(description="Check Cameras", button_style="info", icon="camera")
btn_refresh = widgets.Button(description="Refresh", button_style="info", icon="refresh")
auto_refresh = widgets.ToggleButton(description="Auto Refresh", value=False, icon="repeat")
btn_start = widgets.Button(description="Start", button_style="success", icon="play")
btn_stop = widgets.Button(description="Stop & Save", button_style="warning", icon="stop")
btn_discard = widgets.Button(description="Discard Last", button_style="danger", icon="trash")

btn_init_torso.tooltip = "VR mapping: Right controller A"
btn_init_bimanual.tooltip = "VR mapping: Left controller Y"
btn_check_cameras.tooltip = "Dashboard only"
btn_refresh.tooltip = "Dashboard only"
auto_refresh.tooltip = "Dashboard only"
btn_start.tooltip = "VR mapping: Left controller X toggle from idle"
btn_stop.tooltip = "VR mapping: Right controller B or Left controller X toggle while recording"
btn_discard.tooltip = "Dashboard only"

btn_init_torso.on_click(lambda _: run_command("init_torso"))
btn_init_bimanual.on_click(lambda _: run_command("init_bimanual"))
btn_check_cameras.on_click(lambda _: run_command("check_cameras"))
btn_start.on_click(lambda _: run_command("start"))
btn_stop.on_click(lambda _: run_command("stop"))
btn_discard.on_click(lambda _: run_command("discard"))
btn_refresh.on_click(refresh_dashboard)


def _on_auto_refresh(change):
    if change["new"]:
        start_auto_refresh()
        set_log("Auto refresh enabled")
    else:
        stop_auto_refresh()
        set_log("Auto refresh disabled")


auto_refresh.observe(_on_auto_refresh, names="value")

init_controls = widgets.HBox([btn_init_torso, btn_init_bimanual, btn_check_cameras, btn_refresh, auto_refresh])
record_controls = widgets.HBox([btn_start, btn_stop, btn_discard])
controls = widgets.VBox([init_controls, record_controls])
left_panel = widgets.VBox([title_html, subtitle_html, vr_mapping_html, status_html, preflight_html, camera_html, paths_html, log_html])
right_panel = widgets.VBox([widgets.HTML("<b>Camera Preview</b>"), preview_html, raw_state])
layout = widgets.VBox([controls, widgets.HBox([left_panel, right_panel])])

display(layout)
refresh_dashboard()

## 셀 6. wiring sanity check와 backend 연결 확인

마지막 셀은 helper 함수와 client가 정상적으로 연결되어 있는지 빠르게 점검합니다. backend가 이미 떠 있으면 현재 status, preflight, preview 가능 여부, camera diagnostics 메시지를 출력하고, 아직 안 떠 있으면 어떤 명령으로 띄워야 하는지 다시 안내합니다.

문제가 생겼을 때는 이 셀의 출력이 가장 빠른 1차 진단 포인트입니다.

In [ ]:
assert hasattr(client, "get_status")
assert callable(refresh_dashboard)
assert callable(run_command)

print("Dashboard helper wiring is ready.")
try:
    snapshot = client.get_status(include_preview=True)
    print("Backend reachable:", snapshot.get("ok", False))
    print("Status:", snapshot.get("status"))
    print("Preflight:", snapshot.get("preflight", {}).get("message"))
    print("Camera diagnostics:", snapshot.get("camera_diagnostics", {}).get("message"))
    print("Preview available:", bool(snapshot.get("preview_jpeg_base64")))
except Exception as exc:
    print("Backend is not reachable yet:", exc)
    print("Start the backend with:")
    print(f"  python /home/hyunjin/rby1_ws/rby1-data-collection/main.py --data_collect --dashboard_cmd_server {COMMAND_ENDPOINT}")